# 🔥 FireST-GraphNet: Colab 端到端流程

**功能**：
1. ✅ 使用 Google Drive 中已有的原始数据（跳过下载）
2. ✅ 运行数据预处理生成训练数据
3. ✅ 验证数据正确性
4. ✅ 训练模型（AF + BA + Pred）
5. ✅ 保存结果到 Google Drive

**估计时间**：
- ~~下载原始数据~~：**已跳过** ✅
- **首次运行**（生成数据）：5-9 小时
  - 数据预处理：3-5 小时
  - 保存到 Drive：30-60 分钟
  - 训练：2-4 小时
- **后续运行**（复用数据）：2-4 小时 ⚡
  - ~~数据预处理~~：**已跳过** ✅
  - 训练：2-4 小时

**数据复用说明**：

- 首次运行会自动保存生成的数据到 Google Drive---

- 后续训练可直接使用已保存的数据，跳过预处理（节省 3-5 小时）

- ✅ 192 个火灾事件文件夹，每个包含 VIIRS_Day、VIIRS_Night、FirePred、ESRI_LULC

**前提条件**：- ✅ 原始数据已在 `My Drive/FireST-GraphNet/ts-satfire_raw/`（71GB）

## 📋 准备工作

### 1. 确认 Google Drive 中的文件

确保以下文件夹已在 Google Drive：

```
My Drive/
└── FireST-GraphNet/
    ├── FireST-GraphNet-Clean/     # 项目代码根目录
    │   ├── firest_graphnet/       # 必需：模型和训练代码
    │   ├── data_tools/validation/ # 必需：验证脚本
    │   ├── config_run11.yaml      # 必需：训练配置
    │   └── outputs/               # 可选：保存训练结果的位置
    └── ts-satfire_raw/            # 必需：原始数据（71GB）
        ├── 20562846/              # 火灾事件1
        │   ├── VIIRS_Day/
        │   ├── VIIRS_Night/
        │   ├── FirePred/
        │   └── ESRI_LULC/
        ├── 20568194/              # 火灾事件2
        └── ... (共192个事件)
```

### 2. 数据结构验证

原始数据应包含：
- ✅ 192 个火灾事件文件夹（以事件ID命名）
- ✅ 每个事件包含 4 个子文件夹
- ✅ 总计约 10,811 个 GeoTIFF 文件
- ✅ 总大小约 71GB

---

## 🚀 Step 1: 环境设置

In [ ]:
# 检查 GPU
!nvidia-smi

import os
from pathlib import Path
import shutil

print("\n=== Colab 环境信息 ===")
print(f"可用磁盘空间: {shutil.disk_usage('/').free / (1024**3):.1f} GB")
print(f"工作目录: {os.getcwd()}")

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 设置项目路径
PROJECT_PATH = "/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean"
RAW_DATA_PATH = "/content/drive/MyDrive/FireST-GraphNet/ts-satfire_raw"

# 验证路径
assert Path(PROJECT_PATH).exists(), f"❌ 未找到项目文件夹: {PROJECT_PATH}\n请确保已上传 FireST-GraphNet-Clean 到 Google Drive/FireST-GraphNet/"
assert Path(RAW_DATA_PATH).exists(), f"❌ 未找到原始数据: {RAW_DATA_PATH}\n请确保原始数据在 Google Drive/FireST-GraphNet/ts-satfire_raw/"

print(f"✅ 项目路径: {PROJECT_PATH}")
print(f"✅ 原始数据路径: {RAW_DATA_PATH}")

## 📥 Step 2: 验证原始数据（已跳过下载）

In [ ]:
# 使用 Google Drive 中已有的原始数据
DATA_DIR = RAW_DATA_PATH

print("=== 使用已有的原始数据（跳过下载）===\n")
print(f"原始数据路径: {DATA_DIR}")

# 验证数据完整性
raw_data = Path(DATA_DIR)
if not raw_data.exists():
    raise FileNotFoundError(f"❌ 未找到原始数据: {DATA_DIR}")

# 统计事件数量
event_dirs = [d for d in raw_data.iterdir() if d.is_dir() and d.name.isdigit()]
print(f"\n✅ 找到 {len(event_dirs)} 个火灾事件")

# 检查数据类型
sample_event = event_dirs[0] if event_dirs else None
if sample_event:
    subdirs = [d.name for d in sample_event.iterdir() if d.is_dir()]
    print(f"✅ 数据类型: {', '.join(subdirs)}")
    
    # 统计文件数量
    total_files = sum(1 for _ in raw_data.rglob('*.tif'))
    total_size = sum(f.stat().st_size for f in raw_data.rglob('*.tif')) / (1024**3)
    print(f"✅ GeoTIFF 文件: {total_files:,} 个")
    print(f"✅ 总大小: {total_size:.2f} GB")

print(f"\n✅ 数据验证通过！开始预处理...")
print(f"✅ 节省时间：2-4 小时（跳过下载和解压）")

## 🔧 Step 3: 安装预处理依赖

In [ ]:
# 克隆 TS-SatFire 预处理仓库
PREPROCESS_DIR = "/content/TS-SatFire"

if Path(PREPROCESS_DIR).exists():
    shutil.rmtree(PREPROCESS_DIR)

!git clone https://github.com/zhaoyutim/TS-SatFire.git {PREPROCESS_DIR}
print(f"✅ 克隆完成: {PREPROCESS_DIR}")

In [ ]:
# 安装预处理依赖
!pip install -q rasterio==1.3.4 numpy==1.24.3 scikit-image==0.21.0

print("✅ 依赖安装完成")
!python --version
!pip show rasterio numpy scikit-image | grep "Name\|Version"

## 📦 Step 4: 准备训练数据

### 选项 1: 使用已保存的数据（推荐，如果之前已生成）
### 选项 2: 重新生成数据（首次运行或数据有问题）

下一个 cell 会自动检查 Google Drive 上是否已有生成好的数据。

In [ ]:
# 检查 Google Drive 上是否已有生成好的数据
DRIVE_DATA_TRAIN = f"{PROJECT_PATH}/dataset_train"
DRIVE_DATA_VAL = f"{PROJECT_PATH}/dataset_val"

# 检查数据是否存在
data_exists = Path(DRIVE_DATA_TRAIN).exists() and Path(DRIVE_DATA_VAL).exists()
print(f"Google Drive 数据存在: {data_exists}")

USE_EXISTING_DATA = data_exists

required_files = {
    "train": [
        "af_train_img_seqtoseq_alll_1i_1.npy",
        "af_train_label_seqtoseq_alll_1i_1.npy",
        "firepred_train.npy",
        "lulc_train.npy"
    ],
    "val": [
        "af_val_img_seqtoseq_alll_1i_1.npy",
        "af_val_label_seqtoseq_alll_1i_1.npy",
        "firepred_val.npy",
        "lulc_val.npy"
    ]
}

# 根据用户选择执行不同的流程
if USE_EXISTING_DATA:
    print("\n" + "="*60)
    print("⚡ 使用 Google Drive 上已有的数据（跳过预处理）")
    print("="*60 + "\n")

    # 创建本地数据目录
    DATASET_TRAIN = "/content/dataset_train"
    DATASET_VAL = "/content/dataset_val"

    os.makedirs(DATASET_TRAIN, exist_ok=True)
    os.makedirs(DATASET_VAL, exist_ok=True)

    # 从 Drive 复制数据到本地
    print("从 Google Drive 复制数据到 Colab 本地...\n")

    for split, files in required_files.items():
        dest_dir = DATASET_TRAIN if split == "train" else DATASET_VAL
        src_dir = DRIVE_DATA_TRAIN if split == "train" else DRIVE_DATA_VAL

        print(f"{split.upper()} 数据集：")
        for filename in files:
            src = Path(src_dir) / filename
            dest = Path(dest_dir) / filename
            if src.exists():
                shutil.copy2(src, dest)
                size_mb = dest.stat().st_size / (1024**2)
                print(f"  ✅ {filename:40s} {size_mb:8.1f} MB")
            else:
                print(f"  ❌ {filename:40s} NOT FOUND")

    print(f"\n✅ 数据加载完成！")
    print(f"  训练集: {DATASET_TRAIN}")
    print(f"  验证集: {DATASET_VAL}")
    print(f"\n⏩ 已跳过预处理，节省 3-5 小时")
    print(f"\n请直接跳到 Step 5（验证数据）或 Step 6（训练）")

    # 设置标志跳过后续预处理步骤
    SKIP_PREPROCESSING = True
else:
    print("\n" + "="*60)
    print("🏭 将执行完整的预处理流程")
    print("="*60 + "\n")
    SKIP_PREPROCESSING = False


In [ ]:
# 设置预处理工作目录
os.chdir(PREPROCESS_DIR)

print(f"=== 预处理配置 ===\n")
print(f"预处理脚本目录: {PREPROCESS_DIR}")
print(f"原始数据路径: {DATA_DIR}")
print(f"\n检查原始数据...")

# 验证原始数据路径
raw_data = Path(DATA_DIR)
if raw_data.exists():
    event_dirs = [d for d in raw_data.iterdir() if d.is_dir() and d.name.isdigit()]
    print(f"✅ 找到 {len(event_dirs)} 个火灾事件")
    
    # 显示示例事件结构
    if event_dirs:
        sample = event_dirs[0]
        subdirs = [d.name for d in sample.iterdir() if d.is_dir()]
        print(f"✅ 示例事件 ({sample.name}): {', '.join(subdirs)}")
else:
    print(f"❌ 数据路径不存在: {DATA_DIR}")

print(f"\n准备运行预处理脚本...")

In [ ]:
# 配置 TS-SatFire 脚本的数据路径
print("=== 配置预处理脚本数据路径 ===\n")

# TS-SatFire 脚本可能期望数据在特定位置
# 方法1: 创建软链接指向实际数据（推荐）
import subprocess

link_path = Path(PREPROCESS_DIR) / "data"

print(f"创建软链接: {link_path} -> {DATA_DIR}\n")

# 删除旧链接（如果存在）
if link_path.exists() and link_path.is_symlink():
    link_path.unlink()
    print("✅ 删除旧链接")

# 创建新链接
if not link_path.exists():
    result = subprocess.run(
        ["ln", "-s", str(DATA_DIR), str(link_path)],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"✅ 软链接创建成功")
        print(f"   {link_path} -> {DATA_DIR}")
    else:
        print(f"⚠️  软链接创建失败: {result.stderr}")
        print(f"   将尝试使用环境变量...")
        os.environ['TS_SATFIRE_DATA_DIR'] = str(DATA_DIR)
        print(f"✅ 设置环境变量: TS_SATFIRE_DATA_DIR={DATA_DIR}")

# 验证数据可访问
print(f"\n验证数据访问:")
if link_path.exists() or Path(DATA_DIR).exists():
    test_events = list(Path(DATA_DIR).glob("*/"))[:3]
    print(f"✅ 可访问，示例事件: {[e.name for e in test_events]}")
else:
    print(f"❌ 数据不可访问")

print(f"\n💡 提示: 如果预处理失败并提示找不到数据，")
print(f"   需要手动修改 TS-SatFire 脚本中的数据路径为: {DATA_DIR}")

### 重要参数说明

⚠️ **如果上面设置了 `USE_EXISTING_DATA = True`，请跳过此部分，直接跳到 Step 5**

- `-ts 1`：时间跨度（必须一致）

- `-it 1`：迭代次数（必须一致）### 4.1 生成 AF 数据（Active Fire）

- `-uc af/ba`：使用情况（AF 或 BA）

- `-mode train/val/test`：数据集类型---


**关键**：所有脚本必须使用相同的 `-ts` 和 `-it` 参数！

In [ ]:
%%time
# 生成训练集 AF 数据（约 1.5-2 小时）
print("生成训练集 AF 数据...\n")
print(f"数据源: {DATA_DIR}")
print(f"输出目录: {PREPROCESS_DIR}/dataset/\n")

# ===== 1. 创建输出目录（TS-SatFire 脚本期望的相对路径） =====
# TS-SatFire 脚本会输出到 ./dataset/dataset_train 等相对路径
!mkdir -p {PREPROCESS_DIR}/dataset/dataset_train
!mkdir -p {PREPROCESS_DIR}/dataset/dataset_val
# !mkdir -p {PREPROCESS_DIR}/dataset/dataset_test  # 可选：如果需要生成 test 集时启用

print("✅ 输出目录已创建\n")
print(f"   - {PREPROCESS_DIR}/dataset/dataset_train")
print(f"   - {PREPROCESS_DIR}/dataset/dataset_val\n")

# ===== 2. 备份原始脚本（首次运行） =====
!cp dataset_gen_afba.py dataset_gen_afba.py.backup 2>/dev/null || true
!cp dataset_gen_pred.py dataset_gen_pred.py.backup 2>/dev/null || true

# ===== 3. 安全替换硬编码路径（只在需要时执行） =====
import subprocess

# 检查并替换 dataset_gen_afba.py
result = subprocess.run(
    ["grep", "-q", "/home/z/h/zhao2/CalFireMonitoring/data/", "dataset_gen_afba.py"],
    capture_output=True
)
if result.returncode == 0:
    !sed -i "s|/home/z/h/zhao2/CalFireMonitoring/data/|./data/|g" dataset_gen_afba.py
    print("✅ 已修改 dataset_gen_afba.py 路径")
else:
    print("⏭️  dataset_gen_afba.py 路径已正确，跳过修改")

# 检查并替换 dataset_gen_pred.py
result = subprocess.run(
    ["grep", "-q", "/home/z/h/zhao2/CalFireMonitoring/data/", "dataset_gen_pred.py"],
    capture_output=True
)
if result.returncode == 0:
    !sed -i "s|/home/z/h/zhao2/CalFireMonitoring/data/|./data/|g" dataset_gen_pred.py
    print("✅ 已修改 dataset_gen_pred.py 路径")
else:
    print("⏭️  dataset_gen_pred.py 路径已正确，跳过修改")

print()

# ===== 4. 运行生成脚本 =====
!python dataset_gen_afba.py -mode train -ts 1 -it 1 -uc af

print("\n✅ 训练集 AF 数据生成完成")


In [ ]:
%%time
# 生成验证集 AF 数据（约 15-30 分钟）
print("生成验证集 AF 数据...")
!python dataset_gen_afba.py -mode val -ts 1 -it 1 -uc af

print("\n✅ 验证集 AF 数据生成完成")

### 4.2 生成 BA 数据（Burned Area）

In [ ]:
%%time
# 生成训练集 BA 数据（约 1.5-2 小时）
print("生成训练集 BA 数据...")
!python dataset_gen_afba.py -mode train -ts 1 -it 1 -uc ba

print("\n✅ 训练集 BA 数据生成完成")

In [ ]:
%%time
# 生成验证集 BA 数据（约 15-30 分钟）
print("生成验证集 BA 数据...")
!python dataset_gen_afba.py -mode val -ts 1 -it 1 -uc ba

print("\n✅ 验证集 BA 数据生成完成")

### 4.3 生成 Pred 数据（预测 + firepred/lulc）

In [ ]:
%%time
# 生成训练集 Pred 数据（约 1-2 小时）
print("生成训练集 Pred 数据（包括 firepred 和 lulc）...")
!python dataset_gen_pred.py -mode train -ts 1 -it 1

print("\n✅ 训练集 Pred 数据生成完成")

In [ ]:
%%time
# 生成验证集 Pred 数据（约 10-20 分钟）
print("生成验证集 Pred 数据...")
!python dataset_gen_pred.py -mode val -ts 1 -it 1

print("\n✅ 验证集 Pred 数据生成完成")

## ✅ Step 5: 验证生成的数据

In [ ]:
# 查找生成的数据文件
print("查找生成的 .npy 文件...\n")

import glob

# TS-SatFire 脚本生成在 dataset/dataset_train 和 dataset/dataset_val
output_patterns = [
    f"{PREPROCESS_DIR}/dataset/dataset_train/*.npy",
    f"{PREPROCESS_DIR}/dataset/dataset_val/*.npy",
    f"{PREPROCESS_DIR}/output/**/*.npy",  # 备用：有些版本可能在这里
    f"{PREPROCESS_DIR}/**/*train*.npy",
    f"{PREPROCESS_DIR}/**/*val*.npy",
]

all_npy_files = []
for pattern in output_patterns:
    all_npy_files.extend(glob.glob(pattern, recursive=True))

all_npy_files = list(set(all_npy_files))  # 去重

if all_npy_files:
    print(f"找到 {len(all_npy_files)} 个 .npy 文件：")
    for f in sorted(all_npy_files):
        size_mb = Path(f).stat().st_size / (1024**2)
        print(f"  {Path(f).name:40s} {size_mb:8.1f} MB")
else:
    print("⚠️  未找到 .npy 文件，请检查预处理脚本输出")
    print("\n可能的原因：")
    print("1. 脚本执行失败")
    print("2. 输出目录与预期不同")
    print("3. 数据路径配置错误")

In [ ]:
# 整理新生成的数据（仅当执行了预处理时）
if not SKIP_PREPROCESSING:
    # TS-SatFire 脚本生成的数据位置
    TS_TRAIN_DIR = f"{PREPROCESS_DIR}/dataset/dataset_train"
    TS_VAL_DIR = f"{PREPROCESS_DIR}/dataset/dataset_val"
    
    # 目标位置（与后续训练代码一致）
    DATASET_TRAIN = "/content/dataset_train"
    DATASET_VAL = "/content/dataset_val"
    
    os.makedirs(DATASET_TRAIN, exist_ok=True)
    os.makedirs(DATASET_VAL, exist_ok=True)
    
    print("整理数据文件（从 TS-SatFire 输出目录复制到标准位置）...\n")
    print(f"源路径: {PREPROCESS_DIR}/dataset/")
    print(f"目标路径: /content/\n")
else:
    print("⏩ 跳过（已使用 Google Drive 的数据）\n")

if not SKIP_PREPROCESSING:

# 定义需要的文件
required_files = {
    "train": [
        "af_train_img_seqtoseq_alll_1i_1.npy",
        "af_train_label_seqtoseq_alll_1i_1.npy",
        "firepred_train.npy",
        "lulc_train.npy"
    ],
    "val": [
        "af_val_img_seqtoseq_alll_1i_1.npy",
        "af_val_label_seqtoseq_alll_1i_1.npy",
        "firepred_val.npy",
        "lulc_val.npy"
    ]
}

# 方法1: 直接从 TS-SatFire 生成的目录复制（推荐）
print("\n复制文件到标准位置：\n")
for split in ["train", "val"]:
    src_dir = TS_TRAIN_DIR if split == "train" else TS_VAL_DIR
    dest_dir = DATASET_TRAIN if split == "train" else DATASET_VAL
    
    print(f"{split.upper()} 数据集：")
    
    # 检查源目录是否存在
    if Path(src_dir).exists():
        # 复制所有 .npy 文件
        for src_file in Path(src_dir).glob("*.npy"):
            dest_file = Path(dest_dir) / src_file.name
            shutil.copy2(src_file, dest_file)
            size_mb = dest_file.stat().st_size / (1024**2)
            print(f"  ✅ {src_file.name:40s} {size_mb:8.1f} MB")
    else:
        print(f"  ⚠️  源目录不存在: {src_dir}")
        print(f"  尝试从搜索结果中查找文件...")
        
        # 方法2: 从 all_npy_files 搜索（备用）
        for filename in required_files[split]:
            matches = [f for f in all_npy_files if Path(f).name == filename]
            if matches:
                src = matches[0]
                dest = Path(dest_dir) / filename
                shutil.copy2(src, dest)
                size_mb = dest.stat().st_size / (1024**2)
                print(f"  ✅ {filename:40s} {size_mb:8.1f} MB")
            else:
                print(f"  ❌ {filename:40s} NOT FOUND")

print(f"\n数据整理完成！")
print(f"  训练集: {DATASET_TRAIN}")
print(f"  验证集: {DATASET_VAL}")

# 自动保存到 Google Drive 供下次使用
print("\n" + "="*60)
print("💾 保存数据到 Google Drive（供下次训练复用）")
print("="*60 + "\n")

print("这将需要 30-60 分钟，但能为下次训练节省 3-5 小时！\n")

# 保存训练数据
if Path(DATASET_TRAIN).exists():
    print(f"保存训练数据到: {DRIVE_DATA_TRAIN}")
    if Path(DRIVE_DATA_TRAIN).exists():
        shutil.rmtree(DRIVE_DATA_TRAIN)
    shutil.copytree(DATASET_TRAIN, DRIVE_DATA_TRAIN)
    
    # 显示文件大小
    total_size = sum(f.stat().st_size for f in Path(DRIVE_DATA_TRAIN).glob('*.npy')) / (1024**2)
    print(f"✅ 训练数据已保存 ({total_size:.1f} MB)")

# 保存验证数据
if Path(DATASET_VAL).exists():
    print(f"保存验证数据到: {DRIVE_DATA_VAL}")
    if Path(DRIVE_DATA_VAL).exists():
        shutil.rmtree(DRIVE_DATA_VAL)
    shutil.copytree(DATASET_VAL, DRIVE_DATA_VAL)
    
    # 显示文件大小
    total_size = sum(f.stat().st_size for f in Path(DRIVE_DATA_VAL).glob('*.npy')) / (1024**2)
    print(f"✅ 验证数据已保存 ({total_size:.1f} MB)")

print("\n" + "="*60)
print("✅ 数据已保存！下次训练可直接使用，跳过预处理")
print("="*60)
else:
    print("\n⏩ 跳过保存（已使用 Google Drive 的数据）")

In [ ]:
# 运行验证脚本
import sys
sys.path.insert(0, PROJECT_PATH)

# 检查验证脚本是否存在
verify_script = Path(PROJECT_PATH) / "data_tools" / "validation" / "quick_verify.py"

if verify_script.exists():
    print("运行数据验证...\n")
    !python {verify_script} --train_dir {DATASET_TRAIN} --val_dir {DATASET_VAL}
else:
    print(f"⚠️  验证脚本不存在: {verify_script}")
    print("请确保已将 data_tools/validation/ 上传到 Google Drive")
    
    # 使用简单的手动验证
    print("\n手动验证数据形状...")
    import numpy as np
    
    train_img = np.load(f"{DATASET_TRAIN}/af_train_img_seqtoseq_alll_1i_1.npy", mmap_mode='r')
    train_label = np.load(f"{DATASET_TRAIN}/af_train_label_seqtoseq_alll_1i_1.npy", mmap_mode='r')
    
    print(f"训练图像形状: {train_img.shape}")
    print(f"训练标签形状: {train_label.shape}")
    
    if train_img.shape[0] == train_label.shape[0]:
        print("✅ 图像和标签数量匹配")
    else:
        print("❌ 图像和标签数量不匹配！")

## 🎯 Step 6: 安装训练依赖

In [ ]:
# 安装 FireST-GraphNet 依赖
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric
!pip install -q pyyaml tensorboard scikit-learn matplotlib

print("✅ 训练依赖安装完成")

In [ ]:
# 验证 PyTorch 和 CUDA
import torch

print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 版本: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 内存: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

## 🏋️ Step 7: 训练模型

### 选择训练任务

- **AF (Active Fire Detection)**: 活跃火点检测
- **BA (Burned Area Mapping)**: 燃烧区域映射
- **Pred (Prediction)**: 预测新燃烧区域

In [ ]:
# 设置训练参数
os.chdir("/content")
sys.path.insert(0, PROJECT_PATH)

# 选择配置文件
CONFIG_FILE = f"{PROJECT_PATH}/config_run11.yaml"  # 修改为你的配置文件

# 检查配置文件
if Path(CONFIG_FILE).exists():
    print(f"✅ 配置文件: {CONFIG_FILE}")
    !cat {CONFIG_FILE}
else:
    print(f"❌ 配置文件不存在: {CONFIG_FILE}")
    print("请上传配置文件到 Google Drive")

In [ ]:
# 修改配置文件中的数据路径
import yaml

with open(CONFIG_FILE, 'r') as f:
    config = yaml.safe_load(f)

# 更新数据路径
config['data']['root_train'] = DATASET_TRAIN
config['data']['root_val'] = DATASET_VAL

# 保存临时配置
temp_config = "/content/config_colab.yaml"
with open(temp_config, 'w') as f:
    yaml.dump(config, f)

print(f"✅ 已创建 Colab 配置文件: {temp_config}")
print(f"\n训练数据路径: {config['data']['root_train']}")
print(f"验证数据路径: {config['data']['root_val']}")

In [ ]:
%%time
# 开始训练（2-4 小时，30 epochs）
print("开始训练...\n")

# 导入训练脚本
from firest_graphnet.scripts.train import main as train_main
import argparse

# 设置训练参数
args = argparse.Namespace(
    config=temp_config,
    output_dir="/content/outputs",
    epochs=30,
    batch_size=4,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 开始训练
train_main(args)

print("\n✅ 训练完成！")

## 💾 Step 8: 保存结果到 Google Drive

In [ ]:
# 保存训练结果
OUTPUT_DIR = "/content/outputs"
DRIVE_OUTPUT = f"{PROJECT_PATH}/outputs/colab_run_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"

if Path(OUTPUT_DIR).exists():
    print(f"保存结果到 Google Drive...")
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f"✅ 结果已保存: {DRIVE_OUTPUT}")
    
    # 显示保存的文件
    print("\n保存的文件：")
    !ls -lh {DRIVE_OUTPUT}
else:
    print("⚠️  未找到输出目录")

In [ ]:
# 数据已在 Step 5 中自动保存，此处仅显示位置
if Path(DRIVE_DATA_TRAIN).exists() and Path(DRIVE_DATA_VAL).exists():
    print("✅ 数据已保存在 Google Drive：")
    print(f"  训练集: {DRIVE_DATA_TRAIN}")
    print(f"  验证集: {DRIVE_DATA_VAL}")
    
    # 显示数据大小
    train_size = sum(f.stat().st_size for f in Path(DRIVE_DATA_TRAIN).glob('*.npy')) / (1024**2)
    val_size = sum(f.stat().st_size for f in Path(DRIVE_DATA_VAL).glob('*.npy')) / (1024**2)
    print(f"\n数据大小:")
    print(f"  训练集: {train_size:.1f} MB")
    print(f"  验证集: {val_size:.1f} MB")
    print(f"  总计: {train_size + val_size:.1f} MB")
    
    print(f"\n💡 下次训练时设置 USE_EXISTING_DATA = True 可直接使用这些数据")
else:
    print("⚠️  数据未保存到 Google Drive")
    print("如果想保存以供下次使用，请重新运行 Step 5")

## 📊 Step 9: 可视化结果

In [ ]:
# 绘制训练曲线
import pandas as pd
import matplotlib.pyplot as plt

metrics_file = Path(OUTPUT_DIR) / "metrics.csv"

if metrics_file.exists():
    df = pd.read_csv(metrics_file)
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Loss
    axes[0, 0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0, 0].plot(df['epoch'], df['val_loss'], label='Val')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # F1 Score
    if 'val_f1' in df.columns:
        axes[0, 1].plot(df['epoch'], df['val_f1'])
        axes[0, 1].set_title('Validation F1 Score')
        axes[0, 1].grid(True)
    
    # IoU
    if 'val_iou' in df.columns:
        axes[1, 0].plot(df['epoch'], df['val_iou'])
        axes[1, 0].set_title('Validation IoU')
        axes[1, 0].grid(True)
    
    # Precision & Recall
    if 'val_precision' in df.columns and 'val_recall' in df.columns:
        axes[1, 1].plot(df['epoch'], df['val_precision'], label='Precision')
        axes[1, 1].plot(df['epoch'], df['val_recall'], label='Recall')
        axes[1, 1].set_title('Precision & Recall')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150)
    plt.show()
    
    print("\n最佳验证指标：")
    if 'val_f1' in df.columns:
        best_idx = df['val_f1'].idxmax()
        print(f"Epoch {df.loc[best_idx, 'epoch']}:")
        print(f"  F1: {df.loc[best_idx, 'val_f1']:.4f}")
        if 'val_iou' in df.columns:
            print(f"  IoU: {df.loc[best_idx, 'val_iou']:.4f}")
else:
    print("⚠️  未找到指标文件")

## 🎉 完成！

### 工作流程总结

1. ✅ 在 Colab 上下载了 71GB 原始数据
2. ✅ 生成了训练所需的所有数据（AF + BA + Pred）
3. ✅ 验证了数据正确性
4. ✅ 训练了模型
5. ✅ 保存了结果到 Google Drive

### 优势

- ✅ 无需在本地处理大文件
- ✅ 充分利用 Colab 的免费 GPU
- ✅ 所有操作在一个 notebook 中完成
- ✅ 结果自动保存到 Google Drive

### 下一步

1. 训练其他任务（BA、Pred）
2. 调整超参数
3. 对比不同配置的效果
4. 在测试集上评估

---

**注意**：
- Colab 会话最长 12 小时后会断开
- 建议在预处理和训练完成后及时保存结果
- 如果会话断开，可以重新运行从 Step 6 开始（数据已保存在 Drive）